# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a detailed, step-by-step guide for loading and exploring the FAIR\u005e2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Name: {getattr(metadata, 'name', 'Unknown')}")
print(f"Description: {getattr(metadata, 'description', 'No description found')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The `@id` of record sets, fields, and columns is used to reference each uniquely.

In [ ]:
# List available record sets and their field @id's
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  - Record Set @id: {rs.id}")
    print(f"    Name: {getattr(rs, 'name', 'N/A')}")
    print(f"    Description: {getattr(rs, 'description', 'N/A')}")
    print(f"    Fields / columns:")
    for field in getattr(rs, 'fields', []):
        print(f"      - Field @id: {field.id}")
        print(f"        Name: {getattr(field, 'name', 'N/A')}")
        print(f"        Data type: {getattr(field, 'data_type', 'N/A')}")

Let's display a sample record from each record set.

In [ ]:
# Print a single record example from each record set
for rs in dataset.record_sets:
    print(f"\nRecord Set @id: {rs.id}")
    records_iter = dataset.records(record_set=rs.id)
    try:
        record = next(records_iter)
        print("Sample Record:")
        for k, v in record.items():
            print(f"  {k}: {v}")
    except StopIteration:
        print("  No records found for this record set.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis using record set and field `@id`s only.

In [ ]:
# Gather all record set ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))

Pick one record set with substantial tabular data for further exploratory analysis. (If unsure, inspect `dataframes.keys()`).

In [ ]:
# Choose a main record set for further analysis (replace with relevant @id as seen above)
main_record_set_id = None
for key, df in dataframes.items():
    if df.shape[1] > 3 and df.shape[0] > 5:
        main_record_set_id = key
        break
if main_record_set_id is None:
    raise ValueError("No appropriate tabular record set found. Please check the dataset." )
print(f"We'll use record set @id: {main_record_set_id}")
main_df = dataframes[main_record_set_id]
print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (by its `@id`) for basic analytics such as filtering and normalization.

In [ ]:
# List numeric fields (float/int) by @id (from main record set)
numeric_candidates = []
for field in [f for f in dataset.get_record_set(main_record_set_id).fields]:
    dt = getattr(field, 'data_type', '')
    if dt.lower() in ('float', 'integer', 'number'):
        numeric_candidates.append(field.id)
if not numeric_candidates:
    print("No numeric fields found. Please adjust and retry.")
else:
    print(f"Numeric candidate fields: {numeric_candidates}")

# Pick the first available numeric field for demo:
numeric_field_id = numeric_candidates[0]
print(f"Using numeric field @id for analysis: {numeric_field_id}")

In [ ]:
# EDA: Filtering, normalization, grouping
df = main_df.copy()
# Some fields might be missing or non-numeric, try to coerce
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].median() if df[numeric_field_id].notnull().any() else 0
filtered_df = df[df[numeric_field_id] > threshold].copy()

print(f"Filtered records in field '{numeric_field_id}' > median ({threshold}): {len(filtered_df)} records")
print(filtered_df[[numeric_field_id]].head())

# Normalization
mean = filtered_df[numeric_field_id].mean()
std = filtered_df[numeric_field_id].std() or 1
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping (find a non-numeric field as group candidate):
group_field_id = None
for field in [f for f in dataset.get_record_set(main_record_set_id).fields]:
    dt = getattr(field, 'data_type', '')
    if dt.lower() not in ('float', 'integer', 'number'):
        group_field_id = field.id
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
    print(grouped.head())
else:
    print("No suitable grouping field found or group field not present in data.")

## 5. Visualization
Visualize distributions or relationships (requires matplotlib or seaborn).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of filtered numeric field
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If a group field exists, plot group means
if 'grouped' in locals() and group_field_id:
    plt.figure(figsize=(10,5))
    plot_df = grouped.sort_values(numeric_field_id, ascending=False).head(20)
    sns.barplot(data=plot_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Top 20 group means for {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=60, ha='right')
    plt.show()

## 6. Conclusion
- Using the `mlcroissant` library, we've explored a FAIR\u005e2 standardized dataset from a Croissant schema URL.
- We loaded metadata, discovered record set structures, programmatically referenced fields/columns by their `@id`s, and demonstrated basic EDA workflows.
- This approach ensures robust and reproducible data access across open FAIR datasets.